<a href="https://colab.research.google.com/github/RajBhatta67/measles-rubella-project/blob/main/Measles_and_Rubles_Research_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Checkpoint #1


In [2]:
import pandas as pd
import numpy as np

# Load the datasets from GitHub raw URLs
yearly = pd.read_csv('https://raw.githubusercontent.com/frontiertechinstitute/datasets/main/measles_rubella/measles_rubella_yearly.csv')
monthly = pd.read_csv('https://raw.githubusercontent.com/frontiertechinstitute/datasets/main/measles_rubella/measles_rubella_monthly.csv')
merged = pd.read_csv('https://raw.githubusercontent.com/frontiertechinstitute/datasets/main/measles_rubella/measles_rubella_merged.csv')


In [3]:
yearly.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2380 entries, 0 to 2379
Data columns (total 19 columns):
 #   Column                                                           Non-Null Count  Dtype  
---  ------                                                           --------------  -----  
 0   region                                                           2380 non-null   object 
 1   country                                                          2380 non-null   object 
 2   iso3                                                             2380 non-null   object 
 3   year                                                             2380 non-null   int64  
 4   total_population                                                 2380 non-null   int64  
 5   annualized_population_most_recent_year_only                      2380 non-null   int64  
 6   total_suspected_measles_rubella_cases                            2293 non-null   float64
 7   measles_total                             

In [4]:
# Rename verbose columns to something workable
yearly_df= yearly.rename(columns = {'measles_incidence_rate_per_1000000_total_population': 'measles_incidence_rate',
    'rubella_incidence_rate_per_1000000_total_population': 'rubella_incidence_rate',
    'annualized_population_most_recent_year_only': 'annualized_population',
    'discarded_non_measles_rubella_cases_per_100000_total_population': 'discarded_rate',
    'total_suspected_measles_rubella_cases': 'total_suspected',
    'discarded_cases': 'discarded_total', })

In [5]:
# Check for missing values
yearly_df.isna().sum()

,0
region,0
country,0
iso3,0
year,0
total_population,0
annualized_population,0
total_suspected,87
measles_total,0
measles_lab_confirmed,0
measles_epi_linked,0


In [6]:
# Core columns needed for the regression -- drop rows missing any of these
core_cols = ['measles_total', 'measles_lab_confirmed', 'measles_epi_linked',
             'measles_clinical', 'rubella_total', 'rubella_lab_confirmed',
             'rubella_epi_linked', 'rubella_clinical', 'total_population']

yearly_df = yearly_df.dropna(axis = 0, how = 'any', subset = core_cols)

In [8]:
# Flag which period each row falls into
def get_period(year):
    if 2012 <= year <= 2019:
        return 'core'
    elif 2020 <= year <= 2021:
        return 'covid_disruption'
    elif year == 2025:
        return 'partial_year'
    else:
        return 'post_covid_recovery'

yearly_df['period_flag'] = yearly_df['year'].apply(get_period)

In [9]:
# Build the measles rows
measles_df = yearly_df[['country', 'iso3', 'region', 'year', 'period_flag', 'total_population']].copy()
measles_df['disease'] = 'measles'
measles_df['total_cases'] = yearly_df['measles_total']
measles_df['lab_confirmed'] = yearly_df['measles_lab_confirmed']
measles_df['epi_linked'] = yearly_df['measles_epi_linked']
measles_df['clinical'] = yearly_df['measles_clinical']
measles_df['incidence_rate'] = yearly_df['measles_incidence_rate']

# Build the rubella rows
rubella_df = yearly_df[['country', 'iso3', 'region', 'year', 'period_flag', 'total_population']].copy()
rubella_df['disease'] = 'rubella'
rubella_df['total_cases'] = yearly_df['rubella_total']
rubella_df['lab_confirmed'] = yearly_df['rubella_lab_confirmed']
rubella_df['epi_linked'] = yearly_df['rubella_epi_linked']
rubella_df['clinical'] = yearly_df['rubella_clinical']
rubella_df['incidence_rate'] = yearly_df['rubella_incidence_rate']

# Stack them into one long DataFrame
tidy_df = pd.concat([measles_df, rubella_df], ignore_index = True)

In [12]:
#New variable which I think will align w/ my research question
tidy_df['lab_share'] = tidy_df['lab_confirmed'] / tidy_df['total_cases']

The research question im leaning towards:
To what extent is the reported divergence between measles and rubella incidence across 193 countries (2012–2024) attributable to rubella incidence being more dependent on a country’s lab-confirmation share than measles incidence is?


Checkpoint #2
